In [0]:
from scripts.utils import silver_utils
print(dir(silver_utils))

In [0]:
from pyspark.sql.functions import (
    col, when, sum, lit, trim, lower, upper,
    to_date, current_timestamp,round,datediff,unix_timestamp,expr
)
from datetime import datetime
import uuid,logging

In [0]:
df=spark.read.table("workspace.ecommerce_bronze.df_orders")
logging.info('table read successfully')

In [0]:
silver_utils.check_schema(df)

In [0]:
df = silver_utils.cast_types(df,{
    "order_id" : "string",
    "customer_id" : "string",
    "order_status" : "string",
    "order_purchase_timestamp" : "timestamp",
    "order_approved_at" : "timestamp",
    "order_delivered_timestamp" : "timestamp",
    "order_estimated_delivery_date" : "date"
})

In [0]:
silver_utils.null_profiling(df,"df_orders")
df= silver_utils.handle_nulls_drop(df,["order_id","customer_id","order_purchase_timestamp"])


In [0]:
df= silver_utils.handle_duplicates(df,["order_id"])

In [0]:
df= silver_utils.standardize_strings(df,{
    "order_id" : lambda c : trim(c),
    "order_status" : lambda c : lower(trim(c)),
    "customer_id" : lambda c : trim(c)
})

In [0]:
df_customers = spark.read.table("workspace.ecommerce_bronze.df_customers")
df_with_check = (df.alias("orders").join(df_customers.select("customer_id").alias("customers"),col("orders.customer_id") == col("customers.customer_id"), "left"))

#referential integrity
checks_ref_customers = [
    (
        "customers.customer_id",
        "customer_id exists in df_customers",
        col("customers.customer_id").isNotNull()
    ),
]

dq_ref_customers= silver_utils.build_dq_table(
    spark=spark,
    df=df_with_check,
    checks=checks_ref_customers,
    table_name="df_orders",
    warn_threshold=5.0
)

checks = [
    #nulls
    (
        "order_id",
        "order_id not null",
        col("order_id").isNotNull()
    ),
    (
        "customer_id",
        "pcustomer_id not null",
        col("customer_id").isNotNull()
    ),
    (
        "order_purchase_timestamp",
        "order_purchase_timestamp not null",
        col("order_purchase_timestamp").isNotNull()
    ),

    #business rules
    (
        "order_approved_at",
        "order_approved_at >= order_purchase_timestamp",
        (col("order_approved_at").isNotNull() & col("order_purchase_timestamp").isNotNull() & (col("order_approved_at") >= col("order_purchase_timestamp")))
    ),
    (
        "order_delivered_timestamp",
        "order_delivered_timestamp >= order_approved_at",
        (col("order_delivered_timestamp").isNotNull() & col("order_approved_at").isNotNull() & (col("order_delivered_timestamp") >= col("order_approved_at")))
    ),
    (
        "order_estimated_delivery_date",
        "order_estimated_delivery_date >= order_purchase_timestamp",
        ( col("order_estimated_delivery_date").isNotNull() & col("order_purchase_timestamp").isNotNull() & (col("order_estimated_delivery_date").cast("timestamp") >= col("order_purchase_timestamp")))
        )
]

dq_table = silver_utils.build_dq_table(
    spark=spark,
    df=df,
    checks=checks,
    table_name="df_orders",
    warn_threshold=5.0
)

#combine all dq results into one table
from functools import reduce
from pyspark.sql import DataFrame

dq_final = reduce(DataFrame.union, [dq_table, dq_ref_customers])

dq_final.write.mode("overwrite").saveAsTable("workspace.ecommerce_silver.df_orders_dq")
dq_saved = spark.read.table("workspace.ecommerce_silver.df_orders_dq")
display(dq_saved)


In [0]:
df = df.withColumn("delivery_delay_days",datediff(col("order_delivered_timestamp"),
    col("order_estimated_delivery_date"))
)


df = df.withColumn("processing_time_hours",
    expr("round((unix_timestamp(order_approved_at) - unix_timestamp(order_purchase_timestamp)) / 3600, 2)")
)

In [0]:
df =silver_utils.add_silver_metadata(df)

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.ecommerce_silver.df_orders")
print("saved successfully !")

In [0]:
%sql
--sanity check
SELECT * FROM workspace.ecommerce_silver.df_orders LIMIT 10